# Nordfalk - first look at the weekly sales extract

Sprint 3, wave 1 scoping. Run against the 12/11 drop in `/mnt/nfk_sftp`.
Numbers in the wave-1 steerco pack come from this notebook. -- SB

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
pd.set_option('display.max_columns', 60)

In [ ]:
RAW = '/mnt/nfk_sftp/2021-11-12/'

salg = pd.read_csv(RAW + 'NF_SALG_UGE.csv', sep=';', encoding='latin-1', decimal=',')
butik = pd.read_csv(RAW + 'NF_BUTIK_STAMDATA.csv', sep=';', encoding='latin-1')
vare = pd.read_csv(RAW + 'NF_VARE_HIERARKI.csv', sep=';', encoding='latin-1')
salg.shape, butik.shape, vare.shape

The extract was one file per drop in wave 1. From the FY23 refresh it comes
split per calendar year (`NF_SALG_UGE_2022.csv`, `NF_SALG_UGE_2023.csv`).

In [ ]:
salg['dato'] = pd.to_datetime(salg['dato'], format='%d/%m/%Y')
salg['butik_id'] = salg['butik_id'].astype(str).str.zfill(4)
salg['region'] = salg['butik_id'].str[:2]
salg.head()

In [ ]:
# Region split. 90xx are depots, they should not be in a store extract at all
# but they are, every week.
salg.groupby('region')['omsaetning_dkk'].sum().sort_values(ascending=False)

In [ ]:
uge = salg.groupby('uge').agg({'antal': 'sum', 'omsaetning_dkk': 'sum'}).reset_index()
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(uge['uge'].astype(str), uge['omsaetning_dkk'] / 1e6)
ax.set_ylabel('mio. DKK inkl. moms')
ax.set_xlabel('uge')
plt.xticks(rotation=90)
plt.show()

Week 52 and week 1 are a single trading period in practice. Finance confirmed
on 12/03 that they report them together, so we do the same.

In [ ]:
# Negative lines are returns. Roughly 1.8% of lines in wave 1.
neg = salg[salg['antal'] < 0]
print(len(neg) / len(salg))
neg.groupby('region')['antal'].sum()

In [ ]:
# Net of VAT. Deposit is handled later in the pipeline, ignore it here.
salg['netto_dkk'] = salg['omsaetning_dkk'] / 1.25
salg['stykpris'] = pd.np.where(salg['antal'] > 0, salg['netto_dkk'] / salg['antal'], np.nan)
salg['stykpris'].describe()

In [ ]:
# Category mix, using the two leading digits of the subcategory.
vare['kategori'] = vare['underkategori'].astype(str).str[:2]
mix = salg.merge(vare[['vare_nr', 'kategori']], on='vare_nr', how='left')
mix.groupby('kategori')['omsaetning_dkk'].sum().sort_values(ascending=False) / 1e6

In [ ]:
# Sanity: stores present in sales but missing from the master.
mangler = pd.DataFrame()
for region, frame in salg.groupby('region'):
    ukendt = set(frame['butik_id']) - set(butik['butik_id'].astype(str).str.zfill(4))
    mangler = mangler.append({'region': region, 'ukendte_butikker': len(ukendt)}, ignore_index=True)
mangler

TODO(SB): 1199 is the HQ test store and turns up with real volumes in weeks
where someone has been testing the till software. Excluded in the pipeline.